In [1]:
import fastf1
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

In [2]:
# Enable FastF1 caching
fastf1.Cache.enable_cache("f1_cache")

# Load FastF1 2024 Australian GP race session
session_2024 = fastf1.get_session(2024, 3, "R")
session_2024.load()

core           INFO 	Loading data for Australian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 19 drivers: ['55', '16', '4', '81', '11', '18', '22', '14', '27', '20', '23', '3', '10', '77', '24', '31', '63', '44', '1']


In [12]:
# Extract lap times
laps_2024 = session_2024.laps[["Driver", "LapTime"]].copy()
laps_2024.dropna(subset=["LapTime"], inplace=True)
laps_2024["LapTime (s)"] = laps_2024["LapTime"].dt.total_seconds()

laps_2024 

,Driver,LapTime,LapTime (s)
0,VER,0 days 00:01:27.458000,87.458
1,VER,0 days 00:01:24.099000,84.099
2,VER,0 days 00:01:23.115000,83.115
4,GAS,0 days 00:01:37.304000,97.304
5,GAS,0 days 00:01:24.649000,84.649
...,...,...,...
993,PIA,0 days 00:01:20.199000,80.199
994,PIA,0 days 00:01:20.754000,80.754
995,PIA,0 days 00:01:20.357000,80.357
996,PIA,0 days 00:01:25.255000,85.255


In [5]:
# 2025 Qualifying Data
qualifying_2025 = pd.DataFrame({
    "Driver": ["Lando Norris", "Oscar Piastri", "Max Verstappen", "George Russell", "Yuki Tsunoda",
               "Alexander Albon", "Charles Leclerc", "Lewis Hamilton", "Pierre Gasly", "Carlos Sainz", "Fernando Alonso", "Lance Stroll"],
    "QualifyingTime (s)": [75.096, 75.180, 75.481, 75.546, 75.670,
                           75.737, 75.755, 75.973, 75.980, 76.062, 76.4, 76.5]
})

In [6]:
qualifying_2025

,Driver,QualifyingTime (s)
0,Lando Norris,75.096
1,Oscar Piastri,75.180
2,Max Verstappen,75.481
3,George Russell,75.546
4,Yuki Tsunoda,75.670
5,Alexander Albon,75.737
6,Charles Leclerc,75.755
7,Lewis Hamilton,75.973
8,Pierre Gasly,75.980
9,Carlos Sainz,76.062


In [7]:
# Map full names to FastF1 3-letter codes
driver_mapping = {
    "Lando Norris": "NOR", "Oscar Piastri": "PIA", "Max Verstappen": "VER", "George Russell": "RUS",
    "Yuki Tsunoda": "TSU", "Alexander Albon": "ALB", "Charles Leclerc": "LEC", "Lewis Hamilton": "HAM",
    "Pierre Gasly": "GAS", "Carlos Sainz": "SAI", "Lance Stroll": "STR", "Fernando Alonso": "ALO"
}

In [8]:
qualifying_2025["DriverCode"] = qualifying_2025["Driver"].map(driver_mapping)
qualifying_2025

,Driver,QualifyingTime (s),DriverCode
0,Lando Norris,75.096,NOR
1,Oscar Piastri,75.180,PIA
2,Max Verstappen,75.481,VER
3,George Russell,75.546,RUS
4,Yuki Tsunoda,75.670,TSU
5,Alexander Albon,75.737,ALB
6,Charles Leclerc,75.755,LEC
7,Lewis Hamilton,75.973,HAM
8,Pierre Gasly,75.980,GAS
9,Carlos Sainz,76.062,SAI


In [ ]:
merged_data = qualifying_2025.merge(laps_2024, left_on="DriverCode", right_on="Driver")
# pd.set_option('display.max_rows', None)  # Show all rows
# pd.set_option('display.max_columns', None)  # Show all columns
# pd.set_option('display.width', None)  # No line wrapping

# print(df)
merged_data

,Driver_x,QualifyingTime (s),DriverCode,Driver_y,LapTime,LapTime (s)
0,Lando Norris,75.096,NOR,NOR,0 days 00:01:29.784000,89.784
1,Lando Norris,75.096,NOR,NOR,0 days 00:01:23.183000,83.183
2,Lando Norris,75.096,NOR,NOR,0 days 00:01:22.656000,82.656
3,Lando Norris,75.096,NOR,NOR,0 days 00:01:22.609000,82.609
4,Lando Norris,75.096,NOR,NOR,0 days 00:01:22.685000,82.685
5,Lando Norris,75.096,NOR,NOR,0 days 00:01:22.632000,82.632
6,Lando Norris,75.096,NOR,NOR,0 days 00:01:22.744000,82.744
7,Lando Norris,75.096,NOR,NOR,0 days 00:01:22.630000,82.630
8,Lando Norris,75.096,NOR,NOR,0 days 00:01:22.727000,82.727
9,Lando Norris,75.096,NOR,NOR,0 days 00:01:22.695000,82.695
